# W5D3 — Attention by Hand — Lab

**Week 5 · Day 3 · NLP Foundations** · Lab

No dataset today. No training, no downloads, no progress bars. Thirty lines of NumPy, and they are
the most important thirty lines of the week.

On Monday you proved that TF-IDF gives `the food was good but the service was bad` and
`the service was good but the food was bad` the same vector. Yesterday you found that embeddings
barely separate them either. This afternoon you build the mechanism that fixes it, from the three
vectors on this morning's slide, and reproduce **`[0.42, 0.16, 0.42]`** and **`[0.84, 0.58]`** to
the decimal.

Then you do the one thing that makes tomorrow inevitable: you feed the same three vectors in
**reverse order** and watch the output not move. Attention alone is still order-blind. The assert
that proves it is the point of the lab.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٣ — الانتباه باليد

**الأسبوع الخامس · اليوم الثالث · أساسيات معالجة اللغة** · معمل

لا بيانات اليوم. ولا تدريب ولا تنزيل ولا أشرطة تقدّم. ثلاثون سطرًا من NumPy، وهي أهمّ ثلاثين سطرًا
في الأسبوع.

برهنت يوم الاثنين أن TF-IDF يعطي `the food was good but the service was bad` و
`the service was good but the food was bad` المتّجه نفسه. ووجدت أمس أن التمثيلات المتّجهية تكاد لا
تفصلهما هي أيضًا. وتبني بعد ظهر اليوم الآلية التي تُصلح ذلك، من المتّجهات الثلاثة على شريحة هذا
الصباح، وتُعيد إنتاج **`[0.42, 0.16, 0.42]`** و**`[0.84, 0.58]`** إلى المنزلة العشرية.

ثم تفعل الشيء الوحيد الذي يجعل الغد محتومًا: تُدخل المتّجهات الثلاثة نفسها **بترتيب معكوس** وتراقب
الخرج لا يتحرّك. فالانتباه وحده ما زال أعمى عن الترتيب. والفحص الذي يُبرهن ذلك هو مقصد المعمل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Implement `softmax` from scratch and say why raw scores cannot be used as weights.
- Show the numerically stable form overflowing where the naive one does, and explain the one-line fix.
- Implement single-query attention in three steps and match the lecture's numbers exactly.
- Change the query and predict which way the weights move before you run it.
- Batch the same function over a matrix of queries without a Python loop.
- Prove that attention is permutation-invariant over its values, and name what must be added.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُنفّذ `softmax` من الصفر وتقول لماذا لا تصلح الدرجات الخام أوزانًا.
- أن تُظهر الصورة المستقرّة عدديًا حيث تطفح الساذجة، وأن تشرح الإصلاح في سطر واحد.
- أن تُنفّذ الانتباه بمُستعلِمٍ واحد في ثلاث خطوات وتُطابق أرقام المحاضرة تمامًا.
- أن تغيّر المُستعلِم وتتنبّأ باتجاه حركة الأوزان قبل التشغيل.
- أن تُجمّع الدالة نفسها على مصفوفة مُستعلِمات بلا حلقة بايثون.
- أن تُبرهن أن الانتباه ثابت تحت تبديل قيمه، وأن تسمّي ما يجب إضافته.

</div>


## About the data

**There is none, and that is deliberate.** Every number in this lab is hard-coded from this
morning's slides:

```
V = [[1, 0],      value vector for token 1
     [0, 1],      token 2
     [1, 1]]      token 3
q  = [1, 0]       the query
```

Three tokens, two dimensions. Small enough that you can compute every intermediate value on paper
and catch your own bug in the second where it happens, which is exactly why a real corpus would be
a liability here. A shape error inside a 12-layer transformer on 12,000 reviews is a two-hour
afternoon; the same error on a 3×2 matrix is a two-second one.

**The one thing to be careful about** is that these numbers are *too* forgiving. Attention on a 3×2
matrix works no matter which axis you softmax over, because both axes are tiny and the shapes still
line up. Task 2.5 batches the function precisely so that the axis becomes checkable, and tomorrow's
lab asserts a shape after every single step for the same reason.

<div dir="rtl" align="right">

## عن البيانات

**لا توجد، وهذا مقصود.** فكل رقم في هذا المعمل مكتوب بثبوت من شرائح هذا الصباح:

```
V = [[1, 0],      متّجه القيمة للرمز الأول
     [0, 1],      الرمز الثاني
     [1, 1]]      الرمز الثالث
q  = [1, 0]       المُستعلِم
```

ثلاثة رموز وبُعدان. وهي صغيرة بما يكفي لتحسب كل قيمة وسيطة على الورق وتُمسك خللك في الثانية التي
يقع فيها، وهذا بالضبط سبب كون المُدوّنة الحقيقية عبئًا هنا. فخطأ شكلٍ داخل محوّلٍ بـ١٢ طبقة على
اثنتي عشرة ألف مراجعة يعني ظهيرةً من ساعتين، والخطأ نفسه على مصفوفة ٣×٢ يعني ثانيتين.

**والشيء الوحيد الذي يجب الحذر منه** أن هذه الأرقام متسامحة **أكثر من اللازم**. فالانتباه على
مصفوفة ٣×٢ يعمل أيًّا كان المحور الذي تُطبّق عليه `softmax`، لأن المحورين صغيران والأشكال تتحاذى على
أي حال. وتُجمّع المهمة ٢٫٥ الدالة لهذا السبب بعينه: ليصير المحور قابلًا للفحص، ويفحص معمل الغد شكلًا
بعد كل خطوة للسبب نفسه.

</div>


## Setup

The only imports are NumPy and Matplotlib. `scikit-learn` appears once, in the warm-up, to
re-establish the problem before you solve it.

<div dir="rtl" align="right">

## الإعداد

لا استيراد إلا NumPy وMatplotlib. ويظهر `scikit-learn` مرّةً واحدة في الإحماء ليُعيد تقرير المشكلة
قبل أن تحلّها.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("scikit-learn", "matplotlib")
seed_everything(42)

import json

import numpy as np
import matplotlib.pyplot as plt

use_course_style()
np.set_printoptions(precision=4, suppress=True)

SEED = 42

# This morning's three value vectors and the query, exactly as they appeared on the slide.
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
QUERY = np.array([1.0, 0.0])
FLIPPED_QUERY = np.array([0.0, 1.0])

# The numbers the lab has to reproduce.
EXPECTED_WEIGHTS = np.array([0.42, 0.16, 0.42])
EXPECTED_OUTPUT = np.array([0.84, 0.58])
EXPECTED_FLIPPED = np.array([0.16, 0.42, 0.42])

print(f"V:\n{V}\nquery: {QUERY}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the problem, one more time  (≈25 min)

Everything here works. Two sentences, identical word sets, opposite meanings:

```
the dog bit the man
the man bit the dog
```

Vectorise both with `TfidfVectorizer` and compare the rows. They are **identical** — the same
result you proved on Monday with a different pair, and it is worth seeing twice, because everything
you build in the next hour exists to make that difference visible.

Change one of the two sentences and watch the vectors separate. Then change it back to a reordering
and watch them collapse together again. Order is invisible to everything you have built so far.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: المشكلة مرّةً أخرى (نحو ٢٥ دقيقة)

كل ما هنا يعمل. جملتان بمجموعتَي كلماتٍ متطابقتين ومعنيين متعاكسين:

```
the dog bit the man
the man bit the dog
```

مثّلهما متّجهيًا بـ`TfidfVectorizer` وقارن الصفّين. فهما **متطابقان** — وهي النتيجة نفسها التي
برهنتها يوم الاثنين بزوج آخر، وتستحقّ أن تُرى مرّتين، لأن كل ما تبنيه في الساعة القادمة موجود
ليُظهِر ذلك الفرق.

غيّر إحدى الجملتين وراقب المتّجهين يتفارقان. ثم أعِدها إلى إعادة ترتيبٍ وراقبهما ينطبقان مرّةً أخرى.
فالترتيب غير مرئيّ لكل ما بنيته حتى الآن.

</div>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

SENTENCES = ["the dog bit the man", "the man bit the dog"]

vectors = TfidfVectorizer().fit_transform(SENTENCES).toarray()
IDENTICAL_TFIDF = np.array_equal(vectors[0], vectors[1])

print(f"vocabulary: {TfidfVectorizer().fit(SENTENCES).get_feature_names_out().tolist()}")
print(f"row 0: {vectors[0]}")
print(f"row 1: {vectors[1]}")
print(f"identical: {IDENTICAL_TFIDF}  |  max difference: {np.abs(vectors[0] - vectors[1]).max()}")
print("\nSubject and object are swapped. The representation cannot see it.")

## Section 2 — Core: six tasks  (≈60 min)

1. `softmax` from scratch, and the slide's `[0.42, 0.16, 0.42]`.
2. The stable version, and the naive one overflowing on `[1000, 1001]`.
3. `attention(q, V)` in three steps, and the slide's `[0.84, 0.58]`.
4. Flip the query, and confirm the weights flip with it.
5. Batch it over a matrix of queries, with no Python loop.
6. Reverse the values, and prove the output does not move.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `softmax` من الصفر، و`[0.42, 0.16, 0.42]` التي على الشريحة.
٢. الصورة المستقرّة، والساذجة تطفح على `[1000, 1001]`.
٣. `attention(q, V)` في ثلاث خطوات، و`[0.84, 0.58]` التي على الشريحة.
٤. اقلب المُستعلِم، وتأكّد أن الأوزان تنقلب معه.
٥. جمّعها على مصفوفة مُستعلِمات بلا حلقة بايثون.
٦. اعكس القيم، وبرهِن أن الخرج لا يتحرّك.

</div>


### Task 2.1 — softmax, and why the scores cannot be the weights

A weight has to be positive and the set has to sum to 1. Raw dot products are neither: they can be
negative, and they sum to whatever they sum to. Softmax is exactly the function that fixes both —
exponentiate, then divide by the total.

Implement it, then run it on `[1, 0, 1]`, which is the score vector from this morning's worked
example. You get **`[0.42, 0.16, 0.42]`** — the slide's numbers, and they sum to exactly 1.

Note what it did to the equal scores: the two `1`s got the same weight. A softmax never breaks a
tie, and that is a property you will rely on in task 2.6.

<div dir="rtl" align="right">

### المهمة ٢٫١ — softmax، ولماذا لا تصلح الدرجات أوزانًا

يجب أن يكون الوزن موجبًا وأن تجمع المجموعة إلى واحد. والجداءات القياسية الخام ليست كذلك: فقد تكون
سالبة، وتجمع إلى ما تجمع إليه. وsoftmax هي بالضبط الدالة التي تُصلح الأمرين — تأسيسٌ أُسّي ثم قسمة
على المجموع.

نفّذها ثم شغّلها على `[1, 0, 1]`، وهي متّجه الدرجات من مثال هذا الصباح المحلول. فتحصل على
**`[0.42, 0.16, 0.42]`** أرقام الشريحة، وهي تجمع إلى واحد تمامًا.

ولاحظ ما فعلته بالدرجتين المتساويتين: أخذ الواحدان الوزن نفسه. فـsoftmax لا تفكّ تعادلًا أبدًا، وهذه
خاصّية ستعتمد عليها في المهمة ٢٫٦.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Exponentiate every score with np.exp, then divide by the sum of the exponentials.
# 2) Two lines. Do not reach for scipy — writing it is the exercise.
# 3) Print the result rounded to 2 decimals and check the sum with np.sum.
# Search: "numpy softmax from scratch exp sum"
# https://numpy.org/doc/stable/reference/generated/numpy.exp.html
#
# ١) أسّس كل درجة بـ`np.exp`، ثم اقسم على مجموع الأُسّيات.
# ٢) سطران. ولا تمدّ يدك إلى scipy — فكتابتها هي التمرين.
# ٣) اطبع الناتج مُدوَّرًا إلى منزلتين وافحص المجموع بـ`np.sum`.
# ابحث عن: "numpy softmax from scratch exp sum"
# https://numpy.org/doc/stable/reference/generated/numpy.exp.html
# ────────────────────────────────────────────────────────────────────

    # TODO: Exponentiate the scores and divide by their total.
    # مهمة: أسّس الدرجات واقسم على مجموعها.
SLIDE_SCORES = np.array([1.0, 0.0, 1.0])
# TODO: Run softmax on SLIDE_SCORES, print the weights to 2 decimals and their sum.
# مهمة: شغّل softmax على `SLIDE_SCORES`، واطبع الأوزان إلى منزلتين ومجموعها.

### Task 2.2 — the stable version, and the overflow it prevents

Run `softmax_naive` on `[1000, 1001]`. `np.exp(1000)` is larger than the largest float64, so it
returns `inf`, and `inf / inf` is `nan`. Your weights are gone, and NumPy told you with a warning
you were about to scroll past.

The fix is one line: **subtract the maximum score before exponentiating.** Softmax is invariant
under adding a constant to every score — the constant cancels between numerator and denominator —
so subtracting the max changes nothing mathematically and puts the largest exponent at `exp(0) = 1`.

Implement `softmax` with the shift, confirm it returns real numbers on `[1000, 1001]`, and confirm
it agrees with the naive version on `[1, 0, 1]` to 1e-12. This is the version every one of the next
three days uses.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الصورة المستقرّة والطفح الذي تمنعه

شغّل `softmax_naive` على `[1000, 1001]`. فـ`np.exp(1000)` أكبر من أكبر عدد بدقّة مزدوجة، فتعيد
`inf`، و`inf / inf` هي `nan`. فذهبت أوزانك، وأخبرك NumPy بتحذيرٍ كنت على وشك تمريره.

والإصلاح سطر واحد: **اطرح أكبر درجة قبل التأسيس.** فـsoftmax ثابتة تحت إضافة مقدار ثابت إلى كل
درجة — يتلاشى الثابت بين البسط والمقام — فطرح الأكبر لا يغيّر شيئًا رياضيًا ويضع أكبر أُسٍّ عند
`exp(0) = 1`.

نفّذ `softmax` بالإزاحة، وتأكّد أنها تعيد أعدادًا حقيقية على `[1000, 1001]`، وأنها تتّفق مع الساذجة
على `[1, 0, 1]` إلى ١e−١٢. وهذه هي الصورة التي تستخدمها الأيام الثلاثة القادمة كلها.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Same two lines, with scores - scores.max() inside the np.exp.
# 2) To see the naive one fail without a wall of red, wrap the call in
#    np.errstate(over="ignore", invalid="ignore") and test the result with np.isfinite.
# 3) Then compare both implementations on [1, 0, 1] with np.allclose.
# Search: "softmax numerical stability subtract max overflow"
# https://numpy.org/doc/stable/reference/generated/numpy.errstate.html
#
# ١) السطران نفسهما، مع `scores - scores.max()` داخل `np.exp`.
# ٢) ولترى فشل الساذجة بلا جدارٍ أحمر، لُفّ النداء في
#    `np.errstate(over="ignore", invalid="ignore")` وافحص الناتج بـ`np.isfinite`.
# ٣) ثم قارن التنفيذين على `[1, 0, 1]` بـ`np.allclose`.
# ابحث عن: "softmax numerical stability subtract max overflow"
# https://numpy.org/doc/stable/reference/generated/numpy.errstate.html
# ────────────────────────────────────────────────────────────────────

    # TODO: normalise along the same axis.
    # مهمة: اطرح الأكبر على `axis` (مع إبقاء الأبعاد)، ثم أسّس وطبّع على المحور نفسه.
BIG = np.array([1000.0, 1001.0])
# TODO: real weights, and both agreeing on the slide's scores.
# مهمة: واتّفاقهما على درجات الشريحة.

### Task 2.3 — attention, in three steps

Now the function itself. Given a query `q` and a matrix of value vectors `V`:

1. **Score** — one dot product per value vector: `V @ q`. Three numbers.
2. **Weight** — softmax those scores. Three weights, positive, summing to 1.
3. **Mix** — the weighted sum of the value vectors: `weights @ V`.

Run it on this morning's `V` and `q = [1, 0]`. The weights come out `[0.42, 0.16, 0.42]` and the
output comes out **`[0.84, 0.58]`**. Same as the slide, and if your numbers differ the bug is in
step 1 or step 3, not in the softmax you already tested.

That is attention. There is nothing else in it.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الانتباه في ثلاث خطوات

وإلى الدالة نفسها. بمُستعلِمٍ `q` ومصفوفة متّجهات قيمٍ `V`:

١. **الدرجات** — جداء قياسي واحد لكل متّجه قيمة: `V @ q`. ثلاثة أعداد.
٢. **الأوزان** — softmax لتلك الدرجات. ثلاثة أوزان موجبة تجمع إلى واحد.
٣. **الخلط** — المجموع الموزون لمتّجهات القيم: `weights @ V`.

شغّلها على `V` هذا الصباح و`q = [1, 0]`. فتخرج الأوزان `[0.42, 0.16, 0.42]` ويخرج الناتج
**`[0.84, 0.58]`**. مثل الشريحة، وإن اختلفت أرقامك فالخلل في الخطوة الأولى أو الثالثة لا في
`softmax` التي فحصتها.

هذا هو الانتباه. وليس فيه شيء آخر.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Step 1 is one matrix-vector product. V has one value vector per row, so V @ q gives
#    you one score per row — check the length before you go on.
# 2) Step 2 is the softmax you just wrote.
# 3) Step 3 is weights @ V, not V @ weights. One of those has the wrong shape and the
#    other is the answer.
# 4) Return both the output and the weights — the weights are what you inspect.
# Search: "numpy dot product matrix vector weighted sum rows"
# https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
#
# ١) الخطوة الأولى جداء مصفوفة في متّجه. ولـ`V` متّجه قيمةٍ في كل صف، فيعطيك `V @ q`
#    درجةً لكل صف — فافحص الطول قبل أن تُكمل.
# ٢) والخطوة الثانية هي `softmax` التي كتبتها الآن.
# ٣) والخطوة الثالثة `weights @ V` لا `V @ weights`. فإحداهما بشكلٍ خاطئ والأخرى الجواب.
# ٤) أعِد الخرج والأوزان معًا — فالأوزان هي ما تتفحّصه
#    بعد ذلك.
# ابحث عن: "numpy dot product matrix vector weighted sum rows"
# https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
# ────────────────────────────────────────────────────────────────────

    # TODO: sum of the value vectors together with the weights.
    # مهمة: لمتّجهات القيم مع الأوزان.
# TODO: compare each against the slide.
# مهمة: بالشريحة.

### Task 2.4 — flip the query, and the weights follow

Nothing about the value vectors changes. Only the question changes: `q = [0, 1]` instead of
`[1, 0]`.

Predict the answer before you run it. Token 1 is `[1, 0]`, so it now scores 0. Token 2 is `[0, 1]`,
so it now scores 1. The weights should become **`[0.16, 0.42, 0.42]`** — the slide's step 6, and
exactly the earlier weights with the first two swapped.

This is what "dynamic" means, and it is the whole difference from a fixed weighting scheme like
TF-IDF: the same three tokens get different weights depending on what is being asked. Write the
sentence.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — اقلب المُستعلِم فتتبعه الأوزان

لا يتغيّر شيء في متّجهات القيم. وإنما يتغيّر السؤال: `q = [0, 1]` بدل `[1, 0]`.

تنبّأ بالجواب قبل التشغيل. فالرمز الأول `[1, 0]`، فدرجته الآن صفر. والرمز الثاني `[0, 1]`، فدرجته
الآن واحد. ويجب أن تصير الأوزان **`[0.16, 0.42, 0.42]`** — وهي الخطوة السادسة على الشريحة،
والأوزان السابقة نفسها بتبديل الأولين.

وهذا معنى «دِيناميّ»، وهو الفرق كله عن نظام وزنٍ ثابت مثل TF-IDF: فالرموز الثلاثة نفسها تأخذ أوزانًا
مختلفة تبعًا لما يُسأل عنه. اكتب الجملة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Call the same function with FLIPPED_QUERY. Nothing else changes.
# 2) Compare the two weight vectors element by element and say which two swapped.
# 3) Write DYNAMIC_WEIGHTING as one sentence contrasting this with an IDF weight, which
#    is fixed once the corpus is fitted.
#
# ١) نادِ الدالة نفسها بـ`FLIPPED_QUERY`. ولا شيء آخر يتغيّر.
# ٢) قارن متّجهَي الأوزان عنصرًا بعنصر وقل أيّ اثنين تبادلا.
# ٣) اكتب `DYNAMIC_WEIGHTING` جملةً واحدة تقابل هذا بوزن IDF
#    الثابت بعد تدريب المُتَّجِه على المُدوّنة.
# ────────────────────────────────────────────────────────────────────

# TODO: the flipped weights.
# مهمة: شغّل `attention` بالمُستعلِم المقلوب، واطبع متّجهَي الأوزان معًا، وسجّل الأوزان المقلوبة.

### Task 2.5 — batch it, and pin the axis

One query at a time is a teaching device. A real model computes attention for every token as a
query, all at once, which means `Q` is a matrix and the scores are a matrix too: `Q @ V.T`, of shape
`(n_queries, n_values)`.

Rewrite the function to take a `(n_queries, d)` matrix and return a `(n_queries, d)` matrix. Then do
the check that matters: **row 0 of the batched result must equal the single-query result** to 1e-9,
and the row sums of the weight matrix must all be 1.

That second check is the one to keep. Softmax over the wrong axis of a square score matrix produces
a perfectly plausible matrix of the right shape and completely wrong contents — a silent bug, and
the most common one in this material. The row sums are what catches it.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — جمّعها وثبّت المحور

المُستعلِم الواحد في كل مرّة أداة تعليمية. أما النموذج الحقيقي فيحسب الانتباه لكل رمزٍ مُستعلِمًا في
دفعةٍ واحدة، ومعناه أن `Q` مصفوفة وأن الدرجات مصفوفة أيضًا: `Q @ V.T` بشكل
`(n_queries, n_values)`.

أعِد كتابة الدالة لتأخذ مصفوفة `(n_queries, d)` وتعيد مصفوفة `(n_queries, d)`. ثم أجرِ الفحص
المهمّ: **يجب أن يساوي الصف صفر من الناتج المُجمَّع ناتجَ المُستعلِم الواحد** إلى ١e−٩، وأن تكون
مجاميع صفوف مصفوفة الأوزان كلها واحدًا.

والفحص الثاني هو ما يجدر الاحتفاظ به. فتطبيق softmax على المحور الخاطئ لمصفوفة درجات مربّعة يُخرج
مصفوفةً معقولةً تمامًا بالشكل الصحيح ومحتوى خاطئ تمامًا — خللٌ صامت، وهو الأشيع في هذه المادة.
ومجاميع الصفوف هي ما يُمسكه.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Scores for a batch are Q @ V.T — shape (n_queries, n_values). Print the shape
#    before you softmax anything.
# 2) The softmax has to run along the VALUES axis, which is axis=-1 here. Your stable
#    softmax already takes an axis argument.
# 3) The mix is weights @ V again — the shapes work out to (n_queries, d).
# 4) Build Q by stacking QUERY and FLIPPED_QUERY, then compare row 0 with the earlier
#    single-query output using np.allclose.
# Search: "numpy softmax along axis batch attention shapes"
# https://numpy.org/doc/stable/reference/generated/numpy.allclose.html
#
# ١) درجات الدفعة هي `Q @ V.T` بشكل `(n_queries, n_values)`. اطبع الشكل قبل أن تُطبّق
#    softmax على شيء.
# ٢) ويجب أن تعمل softmax على محور **القيم**، وهو `axis=-1` هنا. وsoftmax المستقرّة
#    عندك تأخذ وسيط محور أصلًا.
# ٣) والخلط هو `weights @ V` مرّةً أخرى — فتخرج الأشكال إلى `(n_queries, d)`.
# ٤) ابنِ `Q` برصّ `QUERY` و`FLIPPED_QUERY`، ثم قارن الصف صفر بخرج المُستعلِم الواحد
#    السابق بـ`np.allclose`.
# ابحث عن: "numpy softmax along axis batch attention shapes"
# https://numpy.org/doc/stable/reference/generated/numpy.allclose.html
# ────────────────────────────────────────────────────────────────────

    # TODO: Score every query against every value, softmax along the values axis, and mix.
    # مهمة: قيّم كل مُستعلِم مقابل كل قيمة، وطبّق softmax على محور القيم، ثم اخلط.
Q = np.stack([QUERY, FLIPPED_QUERY])
# TODO: and confirm every weight row sums to 1.
# مهمة: صف أوزان يجمع إلى واحد.

### Task 2.6 — the order-blindness proof

**This is the lab.** Everything else was preparation.

Feed the same query the same three value vectors in **reverse order**: `V[::-1]`. Predict what
happens to the weights and to the output before you run it.

The weights come back as the same three numbers in reverse — of course they do, because each weight
is computed from its own value vector and nothing else. And then step 3 sums them, and **a sum does
not care about order**, so the output is identical. Not close: identical, to the last bit.

Assert it. `attention(q, V) == attention(q, V[::-1])` is the mathematical statement that attention
is **permutation-invariant over its values**, and it means the mechanism you just built cannot tell
`the dog bit the man` from `the man bit the dog` either.

Then write the sentence naming what has to be added. Tomorrow's warm-up is that sentence, in one
line of arithmetic.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — برهان العمى عن الترتيب

**هذا هو المعمل.** وكل ما سبق تحضير.

أعطِ المُستعلِم نفسه متّجهات القيم الثلاثة نفسها **بترتيب معكوس**: `V[::-1]`. وتنبّأ بما يحدث
للأوزان وللخرج قبل التشغيل.

تعود الأوزان الأعداد الثلاثة نفسها معكوسةً — وهذا طبيعي، لأن كل وزن يُحسب من متّجه قيمته وحده لا
غير. ثم تجمعها الخطوة الثالثة، و**المجموع لا يبالي بالترتيب**، فالخرج متطابق. وليس قريبًا بل
متطابقًا إلى آخر بت.

افحص ذلك. فـ`attention(q, V) == attention(q, V[::-1])` هي القول الرياضي بأن الانتباه **ثابت تحت
تبديل قيمه**، ومعناه أن الآلية التي بنيتها الآن لا تستطيع هي أيضًا أن تميّز
`the dog bit the man` من `the man bit the dog`.

ثم اكتب الجملة التي تسمّي ما يجب إضافته. وإحماء الغد هو تلك الجملة في سطرٍ حسابي واحد.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) V[::-1] reverses the rows. Call attention with the same query on it.
# 2) Compare the weight vectors as SETS (sorted), and the outputs directly.
# 3) Use np.array_equal for the outputs, not np.allclose — the claim is that they are
#    identical, and asserting a tolerance would be a weaker statement than the truth.
# 4) Then write ORDER_BLINDNESS naming the missing ingredient.
# Search: "permutation invariance weighted sum attention positional encoding"
# https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html
#
# ١) `V[::-1]` تعكس الصفوف. نادِ `attention` بالمُستعلِم نفسه عليها.
# ٢) قارن متّجهَي الأوزان كـ**مجموعتين** (مرتّبتين)، والخرجين مباشرةً.
# ٣) استخدم `np.array_equal` للخرجين لا `np.allclose` — فالادّعاء أنهما متطابقان،
#    وفحص التسامح قولٌ أضعف من الحقيقة.
# ٤) ثم اكتب `ORDER_BLINDNESS` مسمّيًا المكوّن الغائب.
# ابحث عن: "permutation invariance weighted sum attention positional encoding"
# https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html
# ────────────────────────────────────────────────────────────────────

# TODO: and both outputs, and record whether the outputs are bit-identical.
# مهمة: وسجّل هل الخرجان متطابقان بتًّا ببت.

## Section 3 — Stretch: self-attention, and why the √d is there  (≈30 min)

Two parts.

**(a) Self-attention.** So far `q` came from outside. In self-attention every token produces its own
query, key and value by three learned projections of itself: `Q = X @ Wq`, `K = X @ Wk`,
`V = X @ Wv`. Scores become `Q @ K.T / √d`. Implement `self_attention(X)` with random projection
matrices and confirm the output has the same shape as `X`.

**(b) Where the √d earns its keep.** Take random `Q` and `K` for `d = 2` and again for `d = 64`,
and plot the softmax of the scores with and without the division. At `d = 2` the two look much the
same. At `d = 64` the unscaled version is nearly one-hot: a dot product of 64 random terms has a
standard deviation of about `√64 = 8`, so the scores span tens, and `exp` of a difference of tens is
a rounding error away from a hard maximum.

That is the motivation, measured rather than asserted. A near-one-hot softmax has almost no
gradient, which is a model that trains extremely slowly for a reason nothing in the loss curve
explains.

<div dir="rtl" align="right">

## القسم الثالث — التمديد: الانتباه الذاتي، ولماذا الجذر التربيعي لـd (نحو ٣٠ دقيقة)

جزءان.

**(أ) الانتباه الذاتي.** كان `q` حتى الآن يأتي من الخارج. وفي الانتباه الذاتي يُنتج كل رمزٍ
مُستعلِمه ومفتاحه وقيمته بثلاثة إسقاطات مُتعلَّمة لنفسه: `Q = X @ Wq` و`K = X @ Wk` و`V = X @ Wv`.
وتصير الدرجات `Q @ K.T / √d`. نفّذ `self_attention(X)` بمصفوفات إسقاط عشوائية وتأكّد أن للخرج شكل
`X` نفسه.

**(ب) حيث يكسب الجذر أجره.** خُذ `Q` و`K` عشوائيين لـ`d = 2` ثم لـ`d = 64`، وارسم softmax للدرجات
بالقسمة وبدونها. فتتشابه الصورتان كثيرًا عند `d = 2`. وعند `d = 64` تكاد الصورة غير المقسومة تكون
أحاديةً ساخنة: فجداءٌ قياسي من ٦٤ حدًّا عشوائيًا انحرافه المعياري نحو `√64 = 8`، فتمتدّ الدرجات
عشراتٍ، و`exp` لفرقٍ بالعشرات على بُعد خطأ تدوير من حدٍّ أقصى قاطع.

هذا هو الدافع، مقيسًا لا مُدَّعى. فـsoftmax التي تقارب الأحادية الساخنة يكاد لا يكون لها تدرّج،
وهذا نموذج يتدرّب ببطء شديد لسبب لا يشرحه شيء في منحنى الخسارة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Three projections: Wq, Wk, Wv, each (d_in, d_out), from a seeded default_rng.
# 2) Scores are Q @ K.T divided by np.sqrt(d_out); softmax along axis=-1; output is
#    weights @ V.
# 3) For part (b), draw Q and K of shape (1, d) and (8, d) for d in (2, 64), compute the
#    scores both ways and plot the two softmax outputs as bar charts.
# 4) Report the max weight in each case — that single number is the whole argument.
# Search: "scaled dot product attention why divide by sqrt d_k"
# https://numpy.org/doc/stable/reference/random/generator.html
#
# ١) ثلاثة إسقاطات: `Wq` و`Wk` و`Wv`، كلٌّ منها `(d_in, d_out)`، من `default_rng` مُبذّر.
# ٢) الدرجات هي `Q @ K.T` مقسومةً على `np.sqrt(d_out)`، وsoftmax على `axis=-1`، والخرج
#    `weights @ V`.
# ٣) وللجزء (ب)، اسحب `Q` و`K` بشكل `(1, d)` و`(8, d)` لـ`d` في `(2, 64)`، واحسب
#    الدرجات بالطريقتين وارسم خرجَي softmax عمودَين بيانيّين.
# ٤) واعرض أكبر وزن في كل حالة — فذلك العدد الواحد هو الحجّة كلها.
# ابحث عن: "scaled dot product attention why divide by sqrt d_k"
# https://numpy.org/doc/stable/reference/random/generator.html
# ────────────────────────────────────────────────────────────────────

# TODO: scaling, run it on a small X, and confirm the output shape equals the input shape.
# مهمة: `X` صغير، وتأكّد أن شكل الخرج يساوي شكل الدخل.
# TODO: without the sqrt(d) division, plot all four, and print the maximum weight in each.
# مهمة: `sqrt(d)` وبدونها، وارسم الأربعة، واطبع أكبر وزن في كل حالة.

## Save your artefact

`attention_check.json` — the three verified numbers, the flipped weights, and the order-blindness
result.

**Tomorrow loads this file.** D4's warm-up reloads it, re-runs your `attention` on the same three
vectors to confirm it still gives `[0.84, 0.58]`, and only then adds the positional vectors and
watches the weights change. If today's numbers are not on disk, tomorrow has nothing to compare
against.

<div dir="rtl" align="right">

## احفظ أثرك

`attention_check.json` — الأعداد الثلاثة المُتحقَّق منها، والأوزان المقلوبة، ونتيجة العمى عن الترتيب.

**ويُحمّل الغد هذا الملف.** فإحماء اليوم الرابع يُعيد تحميله، ويُعيد تشغيل `attention` عندك على
المتّجهات الثلاثة نفسها ليتأكّد أنها ما زالت تعطي `[0.84, 0.58]`، وعندها فقط يضيف متّجهات الموضع
ويراقب تغيّر الأوزان. وإن لم تكن أرقام اليوم على القرص فليس عند الغد ما يقارن به.

</div>


In [ ]:
attention_check = {
    "values": V.tolist(),
    "query": QUERY.tolist(),
    "softmax_of_1_0_1": [round(float(w), 4) for w in softmax(SLIDE_SCORES)],
    "weights": [round(float(w), 4) for w in WEIGHTS],
    "output": [round(float(o), 4) for o in OUTPUT],
    "flipped_query": FLIPPED_QUERY.tolist(),
    "flipped_weights": [round(float(w), 4) for w in FLIPPED_WEIGHTS],
    "batched_row_0_matches": bool(ROW_0_MATCHES),
    "order_invariant": bool(ORDER_INVARIANT),
    "reversed_output": [round(float(o), 4) for o in REVERSED_OUTPUT],
    "naive_softmax_overflows": bool(NAIVE_OVERFLOWS),
}

CHECK_PATH = ARTEFACT_DIR / "attention_check.json"
CHECK_PATH.write_text(json.dumps(attention_check, indent=2), encoding="utf-8")

print(json.dumps(attention_check, indent=2))
print(f"\nwrote {CHECK_PATH.name} — tomorrow reloads it before it changes anything")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>


In [ ]:
check(np.allclose(softmax(SLIDE_SCORES), EXPECTED_WEIGHTS, atol=0.005),
      f"softmax([1, 0, 1]) must be {EXPECTED_WEIGHTS.tolist()} to 2 decimals — got "
      f"{softmax(SLIDE_SCORES).round(4).tolist()}",
      f"يجب أن تكون `softmax([1, 0, 1])` هي {EXPECTED_WEIGHTS.tolist()} إلى منزلتين — والناتج "
      f"{softmax(SLIDE_SCORES).round(4).tolist()}")

check_close(float(softmax(SLIDE_SCORES).sum()), 1.0,
            "softmax weights must sum to exactly 1 — if they do not, the division is wrong or the "
            "axis is",
            "يجب أن تجمع أوزان softmax إلى واحد تمامًا — فإن لم تجمع فالقسمة خاطئة أو المحور",
            tol=1e-12)

check(NAIVE_OVERFLOWS and STABLE_SURVIVES,
      f"the naive softmax must overflow on [1000, 1001] while the stable one survives — naive "
      f"finite: {not NAIVE_OVERFLOWS}, stable finite: {STABLE_SURVIVES}. If the naive one did not "
      f"overflow you did not implement the naive one",
      f"يجب أن تطفح softmax الساذجة على `[1000, 1001]` وأن تنجو المستقرّة — الساذجة منتهية: "
      f"{not NAIVE_OVERFLOWS}، والمستقرّة منتهية: {STABLE_SURVIVES}. فإن لم تطفح الساذجة فلم "
      f"تُنفّذ الساذجة")

check(np.allclose(OUTPUT, EXPECTED_OUTPUT, atol=0.005),
      f"attention(q, V) must give {EXPECTED_OUTPUT.tolist()} to 2 decimals — got "
      f"{OUTPUT.round(4).tolist()}. A wrong answer here is step 1 or step 3, since the softmax "
      f"already passed",
      f"يجب أن يُعطي `attention(q, V)` القيمة {EXPECTED_OUTPUT.tolist()} إلى منزلتين — والناتج "
      f"{OUTPUT.round(4).tolist()}. والجواب الخاطئ هنا في الخطوة الأولى أو الثالثة، فقد اجتازت "
      f"softmax أصلًا")

check(np.allclose(FLIPPED_WEIGHTS, EXPECTED_FLIPPED, atol=0.005),
      f"flipping the query to {FLIPPED_QUERY.tolist()} must give weights "
      f"{EXPECTED_FLIPPED.tolist()} — got {FLIPPED_WEIGHTS.round(4).tolist()}",
      f"يجب أن يُعطي قلب المُستعلِم إلى {FLIPPED_QUERY.tolist()} الأوزان "
      f"{EXPECTED_FLIPPED.tolist()} — والناتج {FLIPPED_WEIGHTS.round(4).tolist()}")

check(ROW_0_MATCHES and ROWS_SUM_TO_ONE,
      f"the batched result's row 0 must equal the single-query output to 1e-9 and every weight "
      f"row must sum to 1 — row 0 matches: {ROW_0_MATCHES}, rows sum to 1: {ROWS_SUM_TO_ONE}. A "
      f"failure on the second half means the softmax ran over the wrong axis",
      f"يجب أن يساوي الصف صفر من الناتج المُجمَّع خرجَ المُستعلِم الواحد إلى ١e−٩ وأن يجمع كل صف "
      f"أوزان إلى واحد — الصف صفر مطابق: {ROW_0_MATCHES}، والصفوف تجمع إلى واحد: "
      f"{ROWS_SUM_TO_ONE}. والفشل في الشقّ الثاني يعني أن softmax عملت على المحور الخاطئ")

check(ORDER_INVARIANT,
      f"reversing the value vectors must leave the output bit-identical — got {OUTPUT.tolist()} "
      f"and {REVERSED_OUTPUT.tolist()}. This is the lab's thesis: attention on its own cannot see "
      f"order, which is why tomorrow starts with positional encoding",
      f"يجب أن يُبقي عكس متّجهات القيم الخرجَ متطابقًا بتًّا ببت — والناتج {OUTPUT.tolist()} و"
      f"{REVERSED_OUTPUT.tolist()}. وهذه أطروحة المعمل: الانتباه وحده لا يرى الترتيب، ولهذا يبدأ "
      f"الغد بترميز الموضع")

check(IDENTICAL_TFIDF,
      "the warm-up's two sentences must have identical TF-IDF vectors — if they do not, the pair "
      "is not a reordering of the same words and the day has no problem to solve",
      "يجب أن يكون لجملتَي الإحماء متّجها TF-IDF متطابقان — فإن لم يكونا فليس الزوج إعادة ترتيبٍ "
      "للكلمات نفسها وليست لليوم مشكلة يحلّها")

report()

## What's next

**W5D4 — Build a transformer block.** Tomorrow is assembly, not invention. `attention` is written,
tested and on disk. What gets added is a position for every token — three numbers per token in the
warm-up, a sinusoid in the core — and then the plumbing around the attention you already have:
multiple heads, a residual, a `LayerNorm`, a feed-forward, another residual, another `LayerNorm`.

The graded skill tomorrow is shape discipline: an assert after every step, recorded in
`block_shapes.json`. It is not busywork. A shape bug in a transformer block does not raise — it
broadcasts, silently, and produces a model that trains to a plausible loss and predicts nothing.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٤ — بناء كتلة محوّل.** الغد تركيب لا اختراع. فـ`attention` مكتوبة ومفحوصة وعلى
القرص. والذي يُضاف موضعٌ لكل رمز — ثلاثة أعداد لكل رمز في الإحماء، وجَيبٌ مثلّثي في الأساسي — ثم
السباكة حول الانتباه الذي عندك: رؤوس متعدّدة، وبقيّة (Residual)، و`LayerNorm`، وشبكة تغذية أمامية،
وبقيّة أخرى، و`LayerNorm` أخرى.

والمهارة المُقيَّمة غدًا انضباط الأشكال: فحصٌ بعد كل خطوة، مُسجَّل في `block_shapes.json`. وليس هذا
عملًا زائدًا. فخلل الشكل في كتلة محوّل لا يرفع استثناءً — بل يُبثّ صامتًا، ويُخرج نموذجًا يتدرّب إلى
خسارةٍ معقولة ولا يتنبّأ بشيء.

</div>
